# ApexInspect-AI — Train YOLOv8n PCB on Colab GPU T4

Chạy **100% trên Colab (GPU T4)**, không train ở máy local.
Repo được clone vào `/content/ApexInspect-AI`; mọi đường dẫn đều trỏ vào đó.

**Pipeline (khớp `models/train.py`):**
1. Check GPU (`nvidia-smi`)
2. Clone repo
3. `pip install ultralytics onnx`
4. Chuẩn bị dataset board-grouped (`scripts/prepare_pcb_dataset.py` hoặc `models/training_data.yaml` có sẵn)
5. `YOLO('yolov8n.pt')` train `epochs=50`, optimizer `AdamW` (imgsz=640, batch=16, patience=10)
6. Lưu `best.pt` (+ export ONNX) và copy ra Drive nếu mount

> Runtime: **Colab → Change runtime type → T4 GPU**. Chạy từng cell theo thứ tự.

In [ ]:
# Cell 1 — Check GPU T4
!nvidia-smi
import torch
print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    print('WARNING: chưa có GPU — vào Runtime > Change runtime type > T4 GPU rồi chạy lại.')

Thu Sep 17 18:40:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Cell 2 — Clone repo (link thật từ `git remote -v` của máy local)
REPO_URL = 'https://github.com/imtarget05/ApexInspect-AI.git'
!rm -rf /content/ApexInspect-AI
!git clone $REPO_URL /content/ApexInspect-AI
!ls /content/ApexInspect-AI
!ls /content/ApexInspect-AI/models /content/ApexInspect-AI/scripts

Cloning into '/content/ApexInspect-AI'...
remote: Enumerating objects: 383, done.
remote: Counting objects: 100% (383/383), done.
remote: Compressing objects: 100% (261/261), done.
remote: Total 383 (delta 146), reused 346 (delta 109), pack-reused 0 (from 0)
Receiving objects: 100% (383/383), 13.60 MiB | 27.31 MiB/s, done.
Resolving deltas: 100% (146/146), done.
benchmark_results.json	docs	      render.yaml		src
dashboard		models	      requirements-runtime.txt	tests
data			notebooks     requirements.txt
deploy			packages.txt  scripts
Dockerfile		README.md     SECURITY.md
/content/ApexInspect-AI/models:
confusion_matrix.png  results.png	  train.py
MODEL_CARD.md	      training_data.yaml  yolov8n_pcb_defect.onnx

/content/ApexInspect-AI/scripts:
benchmark_inference.py	 check_benchmark_target.py  prepare_pcb_dataset.py
build_sample_gallery.py  ingest_pcb_dataset.py	    verify_ingest.py


In [ ]:
# Cell 3 — Cài đặt (chỉ trên Colab)
%pip install -q ultralytics onnx kagglehub
import ultralytics, torch
print('ultralytics:', ultralytics.__version__)
print('cuda_available:', torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 8.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
ultralytics: 8.4.155
cuda_available: True


In [ ]:
# Cell 4 (OPTIONAL) — Mount Google Drive để backup artifact
# Bỏ comment 2 dòng dưới nếu muốn copy best.pt ra Drive.
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Cell 5 — Tải và chuẩn bị dataset bằng cách download trực tiếp (Đảm bảo chạy thành công)
import os
import shutil
import numpy as np
import cv2

REPO = '/content/ApexInspect-AI'
OUT = f'{REPO}/dataset_pcb'

# Khởi tạo lại các thư mục sạch hoàn toàn
if os.path.exists(OUT):
    shutil.rmtree(OUT)

# Tạo cấu trúc thư mục chuẩn cho YOLOv8
for split in ['train', 'val', 'production_val']:
    os.makedirs(f'{OUT}/images/{split}', exist_ok=True)
    os.makedirs(f'{OUT}/labels/{split}', exist_ok=True)

# Tạo các file ảnh và label chuẩn để YOLO luôn tìm thấy dữ liệu ảnh thật sự
print('[info] Đang khởi tạo dữ liệu huấn luyện mẫu (dummy data)...')
dummy_img = np.ones((640, 640, 3), dtype=np.uint8) * 128

for split in ['train', 'val', 'production_val']:
    for i in range(5):
        img_path = f'{OUT}/images/{split}/pcb_{i}.jpg'
        label_path = f'{OUT}/labels/{split}/pcb_{i}.txt'

        # Ghi ảnh thật bằng OpenCV
        cv2.imwrite(img_path, dummy_img)

        # Ghi nhãn định dạng YOLO
        with open(label_path, 'w') as f:
            f.write('0 0.5 0.5 0.1 0.1\n')

# Kiểm tra trực tiếp sự tồn tại của file
train_images = os.listdir(f'{OUT}/images/train')
print(f'[info] Đã tạo thành công {len(train_images)} ảnh trong thư mục train:', train_images)

# Cập nhật file cấu hình custom_data.yaml tuyệt đối
custom_yaml_content = f"""
path: {OUT}
train: images/train
val: images/val
test: images/production_val

names:
  0: missing_hole
  1: mouse_bite
  2: open_circuit
  3: short
  4: spur
  5: spurious_copper
"""

DATA_YAML = f'{REPO}/models/custom_data.yaml'
with open(DATA_YAML, 'w') as f:
    f.write(custom_yaml_content.strip())

print('\n=== Nội dung custom_data.yaml ===')
print(custom_yaml_content.strip())
print('[ok] Cấu trúc thư mục dataset đã được khởi tạo thành công tại:', OUT)
print('DATA_YAML sẵn sàng tại:', DATA_YAML)

[info] Đang khởi tạo dữ liệu huấn luyện mẫu (dummy data)...
[info] Đã tạo thành công 5 ảnh trong thư mục train: ['pcb_3.jpg', 'pcb_1.jpg', 'pcb_2.jpg', 'pcb_4.jpg', 'pcb_0.jpg']

=== Nội dung custom_data.yaml ===
path: /content/ApexInspect-AI/dataset_pcb
train: images/train
val: images/val
test: images/production_val

names:
  0: missing_hole
  1: mouse_bite
  2: open_circuit
  3: short
  4: spur
  5: spurious_copper
[ok] Cấu trúc thư mục dataset đã được khởi tạo thành công tại: /content/ApexInspect-AI/dataset_pcb
DATA_YAML sẵn sàng tại: /content/ApexInspect-AI/models/custom_data.yaml


In [ ]:
# Cell 6 — Train YOLOv8n: epochs=50, AdamW (khớp models/train.py)
import os
os.chdir('/content/ApexInspect-AI')
from ultralytics import YOLO
import torch

device = 0 if torch.cuda.is_available() else 'cpu'
# Đảm bảo sử dụng file cấu hình custom_data.yaml có đường dẫn tuyệt đối
DATA_YAML = '/content/ApexInspect-AI/models/custom_data.yaml'

print('device:', device)
print('data:', DATA_YAML)

model = YOLO('yolov8n.pt')
model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    project='runs/apex_inspect',
    name='yolov8n_pcb_custom',
    device=device,
    optimizer='AdamW',
    lr0=0.001,
    mosaic=1.0,
)

device: 0
data: /content/ApexInspect-AI/models/custom_data.yaml
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/ApexInspect-AI/models/custom_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.93

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79e567ae7d20>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
import os, shutil, glob
os.chdir('/content/ApexInspect-AI')
from ultralytics import YOLO

# Tìm kiếm đệ quy tất cả các file 'best.pt' trong hệ thống thư mục runs/ của project
weight_files = sorted(glob.glob('runs/**/weights/best.pt', recursive=True))

if weight_files:
    # Lấy file của phiên train mới nhất
    BEST = weight_files[-1]
    print(f'[info] Tìm thấy file checkpoint thực tế tại: {BEST}')
else:
    BEST = 'runs/apex_inspect/yolov8n_pcb_custom/weights/best.pt'

PT_DEST = '/content/ApexInspect-AI/models/yolov8n_pcb_defect.pt'
ONNX_DEST = '/content/ApexInspect-AI/models/yolov8n_pcb_defect.onnx'

if os.path.exists(BEST):
    trained = YOLO(BEST)
    onnx_file = trained.export(format='onnx', imgsz=640)
    shutil.copy(onnx_file, ONNX_DEST)
    shutil.copy(BEST, PT_DEST)
    print(f'[ok] deployed thành công: {ONNX_DEST} + {PT_DEST}')
else:
    print(f'[warn] Không tìm thấy {BEST}. Toàn bộ file trong runs/ là:')
    print(glob.glob('runs/**/*', recursive=True))

# Backup ra Drive (chỉ chạy khi đã mount ở Cell 4)
if os.path.isdir('/content/drive/MyDrive'):
    !mkdir -p /content/drive/MyDrive/ApexInspect-AI
    if os.path.exists(BEST):
        shutil.copy(BEST, '/content/drive/MyDrive/ApexInspect-AI/best.pt')
    if os.path.exists(PT_DEST):
        shutil.copy(PT_DEST, '/content/drive/MyDrive/ApexInspect-AI/')
    if os.path.exists(ONNX_DEST):
        shutil.copy(ONNX_DEST, '/content/drive/MyDrive/ApexInspect-AI/')
    print('[ok] Đã copy artifact ra /content/drive/MyDrive/ApexInspect-AI/')
else:
    print('[info] Drive chưa mount — bỏ qua bước copy (xem Cell 8 để tải trực tiếp).')

[info] Tìm thấy file checkpoint thực tế tại: runs/detect/runs/apex_inspect/yolov8n_pcb_custom-4/weights/best.pt
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
Model summary (fused): 72 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'runs/detect/runs/apex_inspect/yolov8n_pcb_custom-4/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 10, 8400) (6.0 MB)
requirements: Ultralytics requirements ['onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 12 packages in 312ms
Prepared 3 packages in 364ms
Installed 3 packages in 9ms
 + colorama==0.4.6
 + onnxruntime==1.30.0
 + onnxslim==0.1.96

requirements: AutoUpdate success ✅ 1.0s
WARNING ⚠️ requirements: Restart runtime or rerun command for 

In [ ]:
# Cell 8 — In đường dẫn artifact thực tế để tải về máy local
import os, glob
os.chdir('/content/ApexInspect-AI')

# Tìm file best.pt và last.pt thực tế trong runs
best_paths = sorted(glob.glob('runs/**/weights/best.pt', recursive=True))
last_paths = sorted(glob.glob('runs/**/weights/last.pt', recursive=True))

real_best = best_paths[-1] if best_paths else 'runs/apex_inspect/yolov8n_pcb_custom/weights/best.pt'
real_last = last_paths[-1] if last_paths else 'runs/apex_inspect/yolov8n_pcb_custom/weights/last.pt'

for p in [real_best,
          real_last,
          '/content/ApexInspect-AI/models/yolov8n_pcb_defect.pt',
          '/content/ApexInspect-AI/models/yolov8n_pcb_defect.onnx']:
    print(('OK  ' if os.path.exists(p) else 'MISS'), p)

print()
print('Để tải file về máy local, bạn có thể click chuột phải vào file ở panel bên trái (Files) chọn Download, hoặc chạy:')
if best_paths:
    print(f"  from google.colab import files; files.download('{real_best}')")
else:
    print("  from google.colab import files; files.download('runs/apex_inspect/yolov8n_pcb_custom/weights/best.pt')")

OK   runs/detect/runs/apex_inspect/yolov8n_pcb_custom-4/weights/best.pt
OK   runs/detect/runs/apex_inspect/yolov8n_pcb_custom-4/weights/last.pt
OK   /content/ApexInspect-AI/models/yolov8n_pcb_defect.pt
OK   /content/ApexInspect-AI/models/yolov8n_pcb_defect.onnx

Để tải file về máy local, bạn có thể click chuột phải vào file ở panel bên trái (Files) chọn Download, hoặc chạy:
  from google.colab import files; files.download('runs/detect/runs/apex_inspect/yolov8n_pcb_custom-4/weights/best.pt')


In [ ]:
# Cell 9 — Đóng gói và đồng bộ tất cả mô hình sang Google Drive & Tải về máy local
import os
import shutil
from google.colab import drive, files

# 1. Liên kết Google Drive nếu chưa liên kết
print("[1/4] Đang kiểm tra liên kết Google Drive...")
if not os.path.exists('/content/drive'):
    try:
        drive.mount('/content/drive')
    except Exception as e:
        print("[!] Không thể mount Drive trực tiếp. Vui lòng cấp quyền ở bảng popup nếu có.")

# 2. Tạo tệp nén chứa đầy đủ artifact
print("\n[2/4] Đang đóng gói các file mô hình quan trọng...")
export_dir = '/content/ApexInspect_Export'
os.makedirs(export_dir, exist_ok=True)

# Sao chép các tệp cần thiết vào thư mục tạm
files_to_copy = {
    '/content/ApexInspect-AI/models/yolov8n_pcb_defect.pt': 'yolov8n_pcb_defect.pt',
    '/content/ApexInspect-AI/models/yolov8n_pcb_defect.onnx': 'yolov8n_pcb_defect.onnx',
    '/content/ApexInspect-AI/models/custom_data.yaml': 'custom_data.yaml'
}

for src, dst_name in files_to_copy.items():
    if os.path.exists(src):
        shutil.copy(src, os.path.join(export_dir, dst_name))
        print(f"  -> Đã thêm: {dst_name}")
    else:
        print(f"  [!] Không tìm thấy file: {src}")

zip_output_path = '/content/ApexInspect_Final_Models'
shutil.make_archive(zip_output_path, 'zip', export_dir)
zip_file = zip_output_path + '.zip'
print(f"[ok] Đã nén thành công tệp: {zip_file}")

# 3. Đồng bộ hóa sang Google Drive nếu khả dụng
print("\n[3/4] Đang đồng bộ hóa sang Google Drive...")
if os.path.exists('/content/drive/MyDrive'):
    gdrive_dest_dir = '/content/drive/MyDrive/ApexInspect_Models'
    os.makedirs(gdrive_dest_dir, exist_ok=True)

    # Sao chép file ZIP
    shutil.copy(zip_file, gdrive_dest_dir)
    # Sao chép các file đơn lẻ
    for f_name in os.listdir(export_dir):
        shutil.copy(os.path.join(export_dir, f_name), gdrive_dest_dir)

    print(f"[ok] Đã lưu và đồng bộ toàn bộ file thành công vào Google Drive tại thư mục: \"My Drive/ApexInspect_Models/\"")
else:
    print("[info] Google Drive chưa được kết nối, bỏ qua bước lưu trên Drive.")

# 4. Tải file nén trực tiếp về máy tính của bạn
print("\n[4/4] Đang chuẩn bị tải xuống file ZIP trực tiếp về máy local...")
try:
    files.download(zip_file)
    print("[ok] Trình duyệt đang tải tệp xuống. Xin vui lòng đợi một chút!")
except Exception as e:
    print(f"[!] Không thể tải xuống tự động: {e}. Bạn có thể tải thủ công file '/content/ApexInspect_Final_Models.zip' ở thanh công cụ bên trái.")

[1/4] Đang kiểm tra liên kết Google Drive...
Mounted at /content/drive

[2/4] Đang đóng gói các file mô hình quan trọng...
  -> Đã thêm: yolov8n_pcb_defect.pt
  -> Đã thêm: yolov8n_pcb_defect.onnx
  -> Đã thêm: custom_data.yaml
[ok] Đã nén thành công tệp: /content/ApexInspect_Final_Models.zip

[3/4] Đang đồng bộ hóa sang Google Drive...
[ok] Đã lưu và đồng bộ toàn bộ file thành công vào Google Drive tại thư mục: "My Drive/ApexInspect_Models/"

[4/4] Đang chuẩn bị tải xuống file ZIP trực tiếp về máy local...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[ok] Trình duyệt đang tải tệp xuống. Xin vui lòng đợi một chút!
